# **TFM Project - Machine Learning for Drug Discovery in Neurodegenerative Diseases**
# **[Part 4] Calculation PaDEL descriptors and dataset preparation**

Carla D. Di Monno

In **Part 4**, molecular descriptors are calculated, using PadelPy library, and filtered to remove any redundant ones. The dataset is then prepared for subsequent model building.

---

## **Installing libraries**

In [ ]:
!pip install padelpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 47.7 MB/s eta 0:00:00


## **Downloaded PaDEL descriptor**

In [ ]:
! wget https://raw.githubusercontent.com/carladdm/PaDEL-Descriptor-master/main/padel.zip
! unzip padel.zip

--2026-04-29 12:19:14--  https://raw.githubusercontent.com/carladdm/PaDEL-Descriptor-master/main/padel.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 25768637 (25M) [application/zip]
Saving to: ‘padel.zip’

padel.zip           100%[===================>]  24.57M  --.-KB/s    in 0.1s    

2026-04-29 12:19:14 (184 MB/s) - ‘padel.zip’ saved [25768637/25768637]

Archive:  padel.zip
   creating: PaDEL-Descriptor/
  inflating: __MACOSX/._PaDEL-Descriptor  
  inflating: PaDEL-Descriptor/MACCSFingerprinter.xml  
  inflating: __MACOSX/PaDEL-Descriptor/._MACCSFingerprinter.xml  
  inflating: PaDEL-Descriptor/AtomPairs2DFingerprinter.xml  
  inflating: __MACOSX/PaDEL-Descriptor/._AtomPairs2DFingerprinter.xml  
  inflating: PaDEL-Descriptor/EStateFingerprinter.xml  

## **Importing libraries**

In [ ]:
import pandas as pd
import tempfile
import shutil # To move the file
import padelpy
from sklearn.feature_selection import VarianceThreshold
import os # Import the os module

## **Functions**

### **1 - Calculate fingerprint descriptors**

In [ ]:
def calculate_padel_descriptors(target_name):
  """
  Generate a SMILES file from the original dataset and calculate the PaDEL descriptors.
  """
  print(f"\n--- Processing target: {target} ---")

  # 1. Load the initial dataset for the target
  input_filename = f'{target_name}_bioactivity_data_lipinski_wo_ext_outliers.csv'
  try:
    df_raw = pd.read_csv(input_filename)
    print(f"Uploaded: {input_filename} (Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]})")
  except FileNotFoundError:
    print(f"Error: The file {input_filename} wasn't found. Make sure you've uploaded or generated it.")
    return False

  # Create a temporary directory for the SMILES file for this target
  with tempfile.TemporaryDirectory() as temp_dir:
    # 2. Select SMILES and molecule_chembl_id, then save to a .smi file in the temporary directory
    smi_filename_in_temp = os.path.join(temp_dir, f'{target_name}_molecule.smi')
    df_raw[['canonical_smiles', 'molecule_chembl_id']].to_csv(smi_filename_in_temp, sep='\t', index=False, header=False)
    print(f"SMILES file generated in temp: {smi_filename_in_temp}")

    # 3. Calculate fingerprints: PaDEL descriptors
    descriptors_output_filename_temp = os.path.join(temp_dir, f'{target_name}_descriptors_output.csv')
    final_descriptors_output_filename = f'{target_name}_descriptors_output.csv'

    padelpy.padeldescriptor(
      mol_dir=temp_dir, # Point to the temporary directory that contains only the current SMILES file
      descriptortypes='./PaDEL-Descriptor/PubchemFingerprinter.xml', # PubChem descriptor
      removesalt=True,
      standardizenitro=True,
      fingerprints=True,
      d_file=descriptors_output_filename_temp # Path to the output file within the temporary directory
    )
    print(f"PaDEL descriptors calculated in temp: {descriptors_output_filename_temp}")

    # Move the output file to the desired location (current directory)
    shutil.move(descriptors_output_filename_temp, final_descriptors_output_filename)
    print(f"PaDEL descriptors moved to: {final_descriptors_output_filename}")
  return True

## **2 - Preparing the X and Y Data Matrices. Saving the final dataset**

In [10]:
def prepare_and_save_dataset(target_name):
  """
  Prepare the X and Y data matrices, apply the variance filter, and save the final dataset.
  """
  print(f"\n--- Preparing the dataset for target: {target} ---")

  # 1. Load descriptors (X)
  descriptors_output_filename = f'{target_name}_descriptors_output.csv'
  try:
    df_X = pd.read_csv(descriptors_output_filename)
    print(f"Uploaded: {descriptors_output_filename} (Initial form: {df_X.shape})")
  except FileNotFoundError:
    print(f"Error: The file {descriptors_output_filename} was not found.")
    return

  # 2. Delete the ‘Name’ column
  df_X = df_X.drop(columns=['Name'])
  print(f"X-shaped after removing ‘Name’: {df_X.shape}")

  # 3. Apply the Variance Threshold to remove features with zero variance
  print(f"Applying the variance threshold (threshold=0) to {target}...")
  sel = VarianceThreshold(threshold=0)
  data_X_filtered = sel.fit_transform(df_X)

  # Get the names of the selected columns
  selected_features = df_X.columns[sel.get_support(indices=True)]
  df_X_filtered = pd.DataFrame(data_X_filtered, columns=selected_features)

  print(f"X-shaped pattern after the variance threshold: {df_X_filtered.shape} ({df_X.shape[1] - df_X_filtered.shape[1]} removed features)")

  # 4. Load the original dataset to obtain the Y variable: Yclass () and Yreg (pchembl_value)
  input_filename = f'{target_name}_bioactivity_data_lipinski_wo_ext_outliers.csv'
  try:
    df_raw = pd.read_csv(input_filename)
    df_Yclass = df_raw['bioactivity_class']
    df_Yreg = df_raw['pchembl_value']
    print(f"Variables Yclass and Yreg extracted from the file: {input_filename} (Shape: {df_Yclass.shape} & {df_Yreg.shape})")
  except FileNotFoundError:
    print(f"Error: The file {input_filename} could not be found to extract Y.")
    return

 # Make sure the indices match before concatenating
  df_X_filtered = df_X_filtered.reset_index(drop=True)
  df_Yclass = df_Yclass.reset_index(drop=True)
  df_Yreg = df_Yreg.reset_index(drop=True)

  # 5. Combine X and Y
  df_final_target = pd.concat([df_X_filtered, df_Yclass, df_Yreg], axis=1)
  print(f"Format of the combined X and Yclass/Yreg dataset: {df_final_target.shape} ")

  # 6. Save the final dataset
  output_filename= f'{target_name}_bioactivity_data_Pubchem-fp_wo_filtered.csv'
  df_final_target.to_csv(output_filename, index=False)
  print(f"Final dataset saved as: {output_filename}")

## **How to use this notebook to process multiple targets**

1.  **Define your Targets:**
    *   The `targets` list contains the names of the therapeutic targets you want to process (e.g., `['AChE', 'MAO-B', 'LRRK2']`). Ensure this list is accurate for your needs.

2.  **Upload Bioactivity Data:**
    *   The filenames for your uploaded data must follow a specific convention: `{Target_name}_bioactivity_data_lipinski_wo_ext_outliers.csv` (e.g., `AChE_bioactivity_data_lipinski_wo_ext_outliers.csv`). Make sure your files are named correctly before uploading.

3.  **Automatic Processing:**
    *   Once the files are uploaded, the notebook will iterate through each target:
        *   It will call `calculate_padel_descriptors(target_name, df_raw_target)` to generate a SMILES file for the target and then compute its PubChem fingerprints using PaDEL-Descriptor.
        *   After descriptor calculation, `prepare_and_save_dataset(target_name, df_raw_target)` will be called. This function loads the calculated descriptors, removes features with zero variance, combines them with the `pchembl_value` (Y variable), and saves the final processed dataset as `{Target_name}_bioactivity_data_Pubchem-fp_wo_filtered.csv`.

**To run the process:**

Simply execute the code cells sequentially.

### **Systematic application of these methods to generate 'fingerprints' for multiples targets**

In [12]:
# List of therapeutic targets to be processed
targets = ['AChE', 'MAO-B', 'LRRK2', 'LRRK2_WT', 'LRRK2_G2019S','GSK3B' ]

print(f"Therapeutic targets to be processed: {targets}")

Therapeutic targets to be processed: ['AChE', 'MAO-B', 'LRRK2', 'LRRK2_WT', 'LRRK2_G2019S', 'GSK3B']


In [13]:
for target in targets:
  if calculate_padel_descriptors(target):
    prepare_and_save_dataset(target)

print("\n--- Complete process for calculating PaDEL descriptors and preparing the dataset for all targets ---")


--- Processing target: AChE ---
Uploaded: AChE_bioactivity_data_lipinski_wo_ext_outliers.csv (Rows: 5593, Columns: 9)
SMILES file generated in temp: /tmp/tmptibgwgmp/AChE_molecule.smi
PaDEL descriptors calculated in temp: /tmp/tmptibgwgmp/AChE_descriptors_output.csv
PaDEL descriptors moved to: AChE_descriptors_output.csv

--- Preparing the dataset for target: AChE ---
Uploaded: AChE_descriptors_output.csv (Initial form: (5593, 882))
X-shaped after removing ‘Name’: (5593, 881)
Applying the variance threshold (threshold=0) to AChE...
X-shaped pattern after the variance threshold: (5593, 635) (246 removed features)
Variables Yclass and Yreg extracted from the file: AChE_bioactivity_data_lipinski_wo_ext_outliers.csv (Shape: (5593,) & (5593,))
Format of the combined X and Yclass/Yreg dataset: (5593, 637) 
Final dataset saved as: AChE_bioactivity_data_Pubchem-fp_wo_filtered.csv

--- Processing target: MAO-B ---
Uploaded: MAO-B_bioactivity_data_lipinski_wo_ext_outliers.csv (Rows: 4387, Colum

---